In [ ]:
import json
import joblib
import pandas as pd
import numpy as np
from scipy import stats
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path
from glob import glob
import textwrap

import config
from utils.data import load_dataset
from experiments.modelconfig import ablation_study
from utils.plotting import df_to_gridded_da, plot_geo_panels, generate_latex_table


In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
model_name = "sonati"
# ablation_base_dir = Path("D:/phd/output/tuning/sonati/architecture_ablation/")
ablation_base_dir = Path("F:/PhD/phd_repos/time_imputation/output/tuning/sonati/architecture_ablation/")
# ablation_base_dir = Path("output/tuning/sonati/architecture_ablation/")

# Metrics
min_metrics = ["MAE", "RMSE", "NRMSE"]
max_metrics = ["R2", "PCC", "KGE", "OI"]
metrics_to_plot = ["Total time", "RMSE", "CMIP_diff", "WOA_diff"]

# Valid experiment ids
masking_exp_ids = ["exp" + str(i) for i in range(53, 70)] + ["exp86"]
per_sample_num_map = {0.3: 1, 0.4: 2, 0.5: 3, 0.7: 4, 0.9: 5}

# Plotting utils
attention_name_map = {
    "space_time_attention": "Space-time attention",
    "mha": "Cross-attention",
    "autoencoder": "Autoencoder",
    "encoder_decoder": "Autoencoder",
    "space_time_depth_attention": "Space-time-depth attention",
    "transformer_encoder_layer": "Joint attention",
    "transformer_encoder": "Joint attention",
    "time_space_attention": "Time-space attention",
}

models_palette = sorted(["Cross-attention", "Joint attention", "Autoencoder",
                         "Space-time attention", "Space-time-depth attention",
                         "Time-space attention"])
palette = dict(zip(models_palette, sns.color_palette("tab10", n_colors=len(models_palette))))


In [ ]:
# Load data
df = load_dataset()
baseline_cfg = ablation_study["exp0"]["config"]
test_idx = json.load(open(f"{config.output_dir_splits}/test_train_split.json", "r"))["test_idx"]

In [ ]:
exps_with_10_seeds = [31, 90, 246, 247, 249]

## Data loading utilities

In [ ]:
def get_attr(res, attr_name):
    """Get attribute from hyps.model.cfg entry of the results object, falling back to the baseline config."""
    cfg_stored = res["hyps"]["model"]["cfg"]
    if attr_name in cfg_stored.keys():
        return cfg_stored[attr_name]

    # Rename attributes: Results from before the sphere -> cylinder rename store the old field names
    legacy_name = attr_name.replace("cylinder_", "sphere_")
    if legacy_name in cfg_stored.keys():
        return cfg_stored[legacy_name]
    else:
        return getattr(baseline_cfg, attr_name)


def find_result_files(exp_dir, model_name="sonati"):
    """Find all per-seed result files for an experiment."""
    return sorted(exp_dir.glob(f"model{model_name}_splitfinal_trial*.json"))


# Track which (exp, seed) already printed a missing-file warning, so repeated
# calls across notebook cells don't spam the same warning multiple times
_missing_diff_warned = set()


def load_woa_cmip_diffs(exp_dir, exp_id, model_name="sonati", seed=None, woa=True, cmip=True):
    """Load WOA/CMIP comparison summaries for one experiment, if available.

    Returns ``(summary_dict, raw_woa_df, raw_cmip_df)``
    """
    out = {"WOA_diff": np.nan, "WOA_std": np.nan, "CMIP_diff": np.nan, "CMIP_std": np.nan}
    raw_woa, raw_cmip = None, None
    seed_suffix = f"_seed{seed}" if seed is not None else ""
    warn_key = f"{exp_id}{seed_suffix}"

    if woa:
        # Load WOA differences
        woa_path = exp_dir / f"{model_name}{seed_suffix}_woa_diff_summary_scaled.csv"
        if not woa_path.is_file():
            woa_path = exp_dir / f"{model_name}_woa_diff_summary_scaled.csv"  # fallback
        if woa_path.is_file():
            raw_woa = pd.read_csv(woa_path)
            out["WOA_diff"] = raw_woa["MAE"].mean()
            out["WOA_std"] = raw_woa["std"].mean()
        elif warn_key not in _missing_diff_warned:
            print(f"No WOA diff summary for {warn_key}")
    if cmip:
        # Load CMIP6 differences
        cmip_path = exp_dir / f"{model_name}{seed_suffix}_cmip_diff_summary_scaled.csv"
        if not cmip_path.is_file():
            cmip_path = exp_dir / f"{model_name}_cmip_diff_summary_scaled.csv"  # fallback
        if cmip_path.is_file():
            raw_cmip = pd.read_csv(cmip_path)
            out["CMIP_diff"] = raw_cmip["MAE"].mean()
            out["CMIP_std"] = raw_cmip["std"].mean()
        elif warn_key not in _missing_diff_warned:
            print(f"No CMIP diff summary for {warn_key}")
    _missing_diff_warned.add(warn_key)
    return out, raw_woa, raw_cmip


def load_experiments(exp_list, attr_list, model_name="sonati", base_dir=None,
                     woa=True, cmip=True, per_var=False):
    """Load results for the given experiment IDs, one row per (experiment, seed)."""
    base = Path(base_dir) if base_dir is not None else ablation_base_dir
    rows, pervar_rows, df_woa, df_cmip = [], [], [], []

    # Iterate over given experiment IDs
    for exp_id in exp_list:
        # Check if result files exist
        exp_name = str(exp_id) if str(exp_id).startswith("exp") else f"exp{exp_id}"
        exp_dir = base / exp_name
        if not exp_dir.is_dir():
            print(f"Experiment {exp_name} not found")
            continue

        # Get result file names
        result_files = find_result_files(exp_dir, model_name)
        if not result_files:
            print(f"No result files for {exp_name}")
            continue

        # Load results
        for res_file in result_files:
            res = json.load(open(res_file, "r"))  # load result file
            metrics_key = "test_metrics" if res["model_framework"] == "pytorch" else "metrics_all"
            seed = res.get("seed", 0)

            # Get metrics of interest
            pervar_dict = res[metrics_key]["PerVariable"]
            m = pd.DataFrame([res[metrics_key]["Global"]])
            m["exp_id"] = exp_name
            m["seed"] = seed
            m["train_time"] = res["train_time"]
            m["pred_time"] = res["pred_time"]
            m["stop_epoch"] = res["stop_epoch"]

            # Median across per-variable KGEs (to prevent artificial inflation)
            kge_pervar = [v["KGE"] for v in pervar_dict.values() if not np.isnan(v.get("KGE", np.nan))]
            if kge_pervar:
                m["KGE"] = np.median(kge_pervar)

            # Masking-related fields - attached unconditionally (regardless of attr_list)
            # since the masking-ablation cells below rely on these being present
            if "model" in res["hyps"] and "cfg" in res["hyps"]["model"]:
                m["mask_ratio"] = get_attr(res, "mask_ratio")
                m["transect_mask_orientation"] = get_attr(res, "transect_mask_orientation")
                m["cylinder_mask_radius"] = get_attr(res, "cylinder_mask_radius")
                m["masking_strategies"] = str(get_attr(res, "masking_strategies"))
                m["transect_mask_width"] = str(get_attr(res, "transect_mask_width"))

            # User-requested fields
            for attr in attr_list:
                m[attr] = get_attr(res, attr)

            # WOA/CMIP diffs
            diffs, raw_woa, raw_cmip = load_woa_cmip_diffs(
                exp_dir, exp_name, model_name, seed=seed, woa=woa, cmip=cmip)
            m["WOA_diff"] = diffs["WOA_diff"]
            m["WOA_std"] = diffs["WOA_std"]
            m["CMIP_diff"] = diffs["CMIP_diff"]
            m["CMIP_std"] = diffs["CMIP_std"]
            rows.append(m)

            # Store per-seed WOA/CMIP diffs separately
            if raw_woa is not None:
                raw_woa = raw_woa.copy()
                raw_woa["exp_id"] = exp_name
                raw_woa["seed"] = seed
                df_woa.append(raw_woa)
            if raw_cmip is not None:
                raw_cmip = raw_cmip.copy()
                raw_cmip["exp_id"] = exp_name
                raw_cmip["seed"] = seed
                df_cmip.append(raw_cmip)

            # Per-variable metrics
            if per_var:
                pv = pd.DataFrame(pervar_dict).T.reset_index(names="parameter")
                pv["exp_id"] = exp_name
                pv["seed"] = seed
                pervar_rows.append(pv)

    if not rows:
        raise ValueError(f"No results loaded for any of {exp_list}")

    # Assemble and return results
    results = pd.concat(rows, ignore_index=True)
    if "attention_type" in results.columns:
        results["attention_type"] = results["attention_type"].map(attention_name_map)

    df_woa = pd.concat(df_woa) if df_woa else None
    df_cmip = pd.concat(df_cmip) if df_cmip else None

    if per_var:
        df_pervar = pd.concat(pervar_rows, ignore_index=True) if pervar_rows else pd.DataFrame()
        return results, df_woa, df_cmip, df_pervar
    return results, df_woa, df_cmip

def aggregate_seeds(df, group_cols, metric_cols=None):
    """Collapse per-seed rows to one row per ablation config, providing
    ``mean $\\pm$ std`` for metric columns, if available.
    """
    # Single column name or a list
    group_cols = [group_cols] if isinstance(group_cols, str) else list(group_cols)

    # Auto-detect metrics
    if metric_cols is None:
        metric_cols = df.select_dtypes(include="number").columns.difference(
            group_cols + ["seed"]).tolist()

    # Other columns to not change
    other_cols = [c for c in df.columns if c not in metric_cols + group_cols + ["seed"]]

    # Compute mean and std per group/ablation config
    grouped = df.groupby(group_cols, as_index=False, sort=False, observed=True)[metric_cols]
    agg = grouped.mean()
    std = grouped.std().rename(columns={c: f"{c}_std" for c in metric_cols})
    agg = agg.merge(std, on=group_cols, how="left")

    # Number of seeds that contributed
    n_seeds = df.groupby(group_cols, as_index=False, sort=False, observed=True)["seed"].nunique()
    agg = agg .merge(n_seeds.rename(columns={"seed": "n_seeds"}), on=group_cols, how="left")

    # Pass through other columns (take first value)
    if other_cols:
        first_vals = df.groupby(group_cols, as_index=False, sort=False, observed=True)[other_cols].first()
        agg = agg.merge(first_vals, on=group_cols, how="left")
    return agg


def warn_incomplete_seeds(df, id_col, expected=5):
    """Print any group whose ``n_seeds`` (from ``aggregate_seeds``) is below ``expected``."""
    incomplete = df[df["n_seeds"] < expected]  # get incomplete entries
    if not incomplete.empty:
        # Print incomplete entries
        pairs = list(zip(incomplete[id_col], incomplete["n_seeds"]))
        print(f"Warning: {len(pairs)} group(s) with fewer than {expected} seeds: {pairs}")

In [ ]:
def extract_mean_miss_ratio(exp_id="exp113"):
    """Compute mean missingness ratio of an ablation experiment across its epochs."""
    dir_path = ablation_base_dir / exp_id
    history = json.load(open(dir_path / "sonati_history.json", "r"))

    avg_miss_ratio = np.nan
    if "train_miss_ratio" in history.keys():
        # Get average missingness over epochs
        miss_ratios = history["train_miss_ratio"]
        avg_miss_ratio = np.mean(list(miss_ratios.values()))
    else:
        print(f"No missingness ratios found for {exp_id}")

    return avg_miss_ratio


def load_averaged_predictions(exp_dir, model_name="sonati"):
    """Load and unscale predictions for an experiment, averaged across all available seeds."""
    # Coordinates (do not need inverse-transform)
    COORD_COLUMNS = {"LATITUDE", "LONGITUDE", "LEV_M", "DATEANDTIME"}

    # Collect result file names
    seed_files = sorted(exp_dir.glob(f"{model_name}_seed*_y_pred.csv"))
    if seed_files:
        # Multi-seed run: Collect (pred, true) CSV pairs for each seed
        seeds = sorted(int(f.stem.split("seed")[1].split("_")[0]) for f in seed_files)
        pred_true_paths = [(exp_dir / f"{model_name}_seed{s}_y_pred.csv",
                            exp_dir / f"{model_name}_seed{s}_y_true.csv") for s in seeds]
    else:
        # Single-run naming
        pred_true_paths = [(exp_dir / f"{model_name}_y_pred.csv", exp_dir / f"{model_name}_y_true.csv")]

    scalers = joblib.load(exp_dir / "scalers.joblib")
    imputed_frames = []
    for pred_path, true_path in pred_true_paths:
        # Load true and predicted data
        df_pred = pd.read_csv(pred_path)
        df_true = pd.read_csv(true_path)

        # Fill missing cells with the model prediction (observed cells stay as ground truth)
        df_imputed = df_true.fillna(df_pred)
        for k, v in scalers.items():
            if k in df_pred.columns and k not in COORD_COLUMNS:
                # Unscale parameters (to physical units)
                df_imputed[k] = v.inverse_transform(df_imputed[k].to_numpy().reshape(-1, 1))

        # Cleaning columns
        if "Unnamed: 0" in df_imputed.columns:
            df_imputed = df_imputed.drop("Unnamed: 0", axis=1)
        imputed_frames.append(df_imputed)

    # Average across seeds
    df_avg = pd.concat(imputed_frames).groupby(level=0).mean()
    return df_avg, len(imputed_frames)


In [ ]:
def barplot_seeds(data, x, y, hue=None, order=None, hue_order=None, ax=None,
                  seed_col="seed", legend_title=None, bar_alpha=0.85, **bar_kwargs):
    """Bar plot that shows seed spread, if available."""
    ax = ax or plt.gca()
    group_cols = [c for c in [x, hue] if c is not None]

    # Only show seed spread if a seed column exists and some group has >1 seeds
    multi_seed = (
        seed_col in data.columns
        and data.groupby(group_cols, observed=True)[seed_col].transform("nunique").gt(1).any()
    )

    # Draw bar plot
    sns.barplot(data=data, x=x, y=y, hue=hue, order=order, hue_order=hue_order,
                errorbar="sd" if multi_seed else None,
                capsize=0.12 if multi_seed else 0,
                alpha=bar_alpha if multi_seed else 1.0, ax=ax, **bar_kwargs)

    if multi_seed:
        # Overlay individual seed points on top of the bars
        strip_kwargs = {"palette": "dark:black"} if hue is not None else {"color": "black"}
        sns.stripplot(data=data, x=x, y=y, hue=hue, order=order, hue_order=hue_order,
                      dodge=bool(hue), jitter=0.12, size=4, alpha=0.6,
                      ax=ax, legend=False, **strip_kwargs)
        if hue is not None:
             # Keep only bar plot legend entries (not strip plot)
            handles, labels = ax.get_legend_handles_labels()
            n_hue = len(hue_order) if hue_order else data[hue].nunique()
            ax.legend(handles[:n_hue], labels[:n_hue], title=legend_title or hue)
    return ax


def safe_log_yscale(ax, series):
    """Set a log y-scale on ``ax`` only if ``series`` has a finite, positive value."""
    # Drop non-finite and nan values
    values = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()

    # Apply log scale
    if (values > 0).any():
        ax.set_yscale("log")
        return True
    return False


# Attention type

Ablates the attention mechanism and the optional feature mixer across six attention variant (cross-attention, joint attention, autoencoder, space-time, space-time-depth, time-space). The feature mixer is a small MLP applied before attention; it can operate on features only (`feat`) or features plus missingness mask indicators (`feat_mask`).

## Feature mixer ablation

In [ ]:
# Feature mixer ablation (multi-seed)
exp_ids = [0, 3, 15,   # mha (cross-attention)
           10, 250, 251,  # transformer_encoder (joint attention)
           11, 37, 27,   # autoencoder
           17, 31, 72,   # space_time_attention
           245, 246, 247,  # time_space_attention
           18, 248, 249,  # space_time_depth_attention
           ]
attr_list = ["attention_type", "feature_mixer_input", "feature_mixer"]

results, df_woa, df_cmip, df_pervar_attn = load_experiments(exp_ids, attr_list, per_var=True)

# Define plottable columns
results["Feature mixer"] = np.select(
    [~results["feature_mixer"],
     results["feature_mixer"] & (results["feature_mixer_input"] == "feat"),
     results["feature_mixer"] & (results["feature_mixer_input"] == "feat_mask")],
    ["None", "On features", "On features and masks"], default=""
)
results["Total time"] = results["train_time"] + results["pred_time"]
results["Total time epochs"] = results["Total time"] / results["stop_epoch"]

# Consistent ordering: attention types by mean RMSE, feature-mixer levels fixed
attn_order = results.groupby("attention_type")["RMSE"].mean().sort_values(ascending=False).index.tolist()
mixer_order = ["None", "On features", "On features and masks"]
mixer_palette = dict(zip(mixer_order, ["#D9D9D9", "#E69F00", "#0072B2"]))  # light grey and Okabe-Ito palette (colour-blind save)
hatches = dict(zip(mixer_order, ["", "////", "xxxx"]))

# Iterate over error types
for m in ["RMSE", "KGE", "Total time epochs", "Total time", "CMIP_diff", "WOA_diff"]:
    fig, ax = plt.subplots(figsize=(8, 5))

    # Barplot
    barplot_seeds(results, x="attention_type", y=m, hue="Feature mixer",
                 order=attn_order, hue_order=mixer_order, legend_title="Feature mixer", ax=ax,
                 palette=mixer_palette, bar_alpha=1.0)

    # Add hatching
    plt.rcParams["hatch.linewidth"] = 0.6
    for cont, lvl in zip(ax.containers, mixer_order):
        for bar in cont:
            bar.set_hatch(hatches[lvl])
            bar.set_edgecolor("black")
            bar.set_linewidth(0.5)
    handles = [Patch(facecolor=mixer_palette[l], edgecolor="black", hatch=hatches[l], linewidth=0.5, label=l) for l in mixer_order]
    ax.legend(handles=handles, title="Feature mixer")

    # Labels
    ax.set_xlabel("")
    plt.xticks(rotation=90)
    y_labels = {"Total time epochs": "Total time / epochs", "CMIP_diff": "CMIP6 MAE", "WOA_diff": "WOA MAE"}
    ax.set_ylabel(y_labels.get(m, m))

    plt.tight_layout()
    plt.savefig(f"output/plots/ablation/attention/feature_mixer_{m}.pdf", dpi=1000)
    plt.show()


In [ ]:
# Generate latex results table
results_for_latex = results.copy()
results_for_latex["Attention type"] = results_for_latex["attention_type"]
results_for_latex["Feature mixer"] = np.select(
    [~results["feature_mixer"],
     results["feature_mixer"] & (results["feature_mixer_input"] == "feat"),
     results["feature_mixer"] & (results["feature_mixer_input"] == "feat_mask")],
    ["None", "Features", "Features+masks"], default=""
)

# Aggregate seeds to one row per experiment - generate_latex_table takes the
# first matching row per exp_id, not the mean, so this must happen first
results_for_latex = aggregate_seeds(
    results_for_latex, group_cols=["exp_id", "Attention type", "Feature mixer"])
warn_incomplete_seeds(results_for_latex, id_col="exp_id")

# Per variable results
df_pervar_for_latex = aggregate_seeds(df_pervar_attn, group_cols=["exp_id", "parameter"])
df_pervar_for_latex = df_pervar_for_latex.merge(
    results_for_latex[["exp_id", "Attention type", "Feature mixer"]], on="exp_id"
)

# Sorting
results_for_latex = results_for_latex.sort_values(["Attention type", "Feature mixer"])

# Generate latex table
latex = generate_latex_table(
    df_global=results_for_latex,
    df_pervar=df_pervar_for_latex,
    model_col="exp_id",
    label_cols=["Attention type", "Feature mixer"],
    bold_best=True,
    sideways=True,
)
print(latex)


## Statistical comparison of top attention candidates

Seed-matched pairwise comparison (RMSE, CMIP6 MAE) of the three finalist configurations, all using the `feat_mask`
feature mixer and all run with the extended 10-seed set: space-time attention (exp31),
space-time-depth attention (exp249), and time-space attention (exp247).

In [ ]:
def paired_seed_test(df, exp_a, exp_b, metrics=("RMSE", "CMIP_diff"), seeds=None, n_boot=10000, seed=0):
    """Seed-matched paired comparison of two experiments on one or more metrics."""
    rng = np.random.default_rng(seed)
    sub = df[df["exp_id"].isin([exp_a, exp_b])]
    if seeds is not None:
        # Restrict to a subset of seeds
        sub = sub[sub["seed"].isin(seeds)]

    # One row per seed, one column per exp_id, per metric
    wide = sub.pivot_table(index="seed", columns="exp_id", values=list(metrics))
    rows = []
    for metric in metrics:
        # Drop seeds missing from either experiment
        pair = wide[metric][[exp_a, exp_b]].dropna()
        x, y = pair[exp_a].to_numpy(), pair[exp_b].to_numpy()
        diff = x - y

        # Wilcoxon test
        t_stat, t_p = stats.ttest_rel(x, y)
        try:
            w_stat, w_p = stats.wilcoxon(x, y)
        except ValueError:
            # Wilcoxon fails if all diffs are 0 (or too few samples)
            w_stat, w_p = np.nan, np.nan

        # Bootstrap the seed-level paired differences to get a confidence interval
        boot_means = np.array([
            diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n_boot)
        ])
        ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])

        rows.append({
            "metric": metric,
            "n_seeds": len(diff),
            f"mean_{exp_a}": x.mean(),
            f"mean_{exp_b}": y.mean(),
            "mean_diff (a-b)": diff.mean(),
            "t_p": t_p,
            "wilcoxon_p": w_p,
            "boot_ci_lo": ci_lo,
            "boot_ci_hi": ci_hi,
            "boot_ci_excludes_0": not (ci_lo < 0 < ci_hi),  # confidence interval excluding 0 suggests a real difference
        })

    return pd.DataFrame(rows)


# Significance test on all three finalist pairs (full 10 seeds as all three have the extended seed set)
finalists = ["exp31", "exp249", "exp247"]
pairwise_10 = pd.concat(
    [paired_seed_test(results, a, b, metrics=["RMSE", "CMIP_diff"]).assign(pair=f"{a} vs {b}")
     for a, b in itertools.combinations(finalists, 2)],
    ignore_index=True,
)
display(pairwise_10)

## Attention type ablation (no feature mixer)

In [ ]:
# Times and depths to plot
depth = 0.0
times = [1772, 2012]

# No feature mixer experiments
ATTENTION_EXPERIMENTS = [
    (0, "Cross-attention"),
    (10, "Joint attention"),
    (11, "Autoencoder"),
    (17, "Space-time attention"),
    (18, "Space-time-depth attention"),
    (245, "Time-space attention"),
]

# Feature mixer experiments (features and missingness masks as input)
# ATTENTION_EXPERIMENTS_FEAT_MASK = []

pred_data_attn = {}
for exp_id, label in ATTENTION_EXPERIMENTS:
    file_name = ablation_base_dir / f"exp{exp_id}"
    key = f"{model_name}_exp{exp_id}"

    # Geo maps show the field averaged across all available seeds
    df_imputed, n_seeds_used = load_averaged_predictions(file_name, model_name)
    print(f"{key}: averaged over {n_seeds_used} seed(s)")

    # Add coordinate information
    df_imputed[config.coordinates] = df[config.coordinates]

    # Assemble final df and filter for times/depths of interest
    pred_data_attn[key] = df_imputed[(df_imputed["LEV_M"] == depth) & (df_imputed["DATEANDTIME"].isin(times))]


In [ ]:
# Plot times/depths slices of interest per parameter
col_items_attn = [(f"sonati_exp{exp_id}", label) for exp_id, label in ATTENTION_EXPERIMENTS]

for parameter in config.parameters:
    gridded = {key: df_to_gridded_da(pred_data_attn[key], value_col=parameter) for key, _ in col_items_attn}
    plot_geo_panels(
        get_data=lambda t, m, _g=gridded: _g[m].sel(time=int(t), depth=0.0),
        row_items=[(t, str(t)) for t in times],
        col_items=col_items_attn,
        plot_config={"cmap": "inferno"},
        colorbar_label=config.parameter_name_unit_map[parameter],
        output_path="output/plots/ablation/attention",
        filename=f"imputation_depth_maps_attention_ablation_{parameter}.pdf",
        dpi=1000,
        width=5.67,
        fontsize=7,
        tick_fontsize=5,
        nodate_colour="0.55",
    )


# Masking strategies

Evaluates how the masking strategy *during training* affects generalisation to unseen missing patterns. Three attention types are compared (autoencoder, space-time attention, time-space attention). The effective missingness ratio, i.e. the actual fraction of hidden values, is tracked alongside the nominal mask ratio.

In [ ]:
exp_map = {
    "autoencoder": {
        "masking": {
            "per_sample": ["exp182", "exp183", "exp184", "exp185", "exp186"],
            "transect": ["exp217", "exp223", "exp229"],
            "transect_random": ["exp218", "exp219", "exp220", "exp221", "exp222", "exp224", "exp225", "exp226",
                                "exp227", "exp228", "exp230", "exp231", "exp232", "exp233", "exp234"],
            "cylinder": ["exp187", "exp193", "exp199", "exp205", "exp211"],
            "cylinder_random": ["exp188", "exp189", "exp190", "exp191", "exp192",
                              "exp194", "exp195", "exp196", "exp197", "exp198",
                              "exp200", "exp201", "exp202", "exp203", "exp204",
                              "exp206", "exp207", "exp208", "exp209", "exp210",
                              "exp212", "exp213", "exp214", "exp215", "exp216"],
            "combined": ["exp244"],
            "per_feature": ["exp235", "exp236", "exp237", "exp238", "exp239", "exp240", "exp241", "exp242", "exp243"],
        }
    },
    "space_time_attention": {
        "masking": {
            "per_sample": ["exp73", "exp91", "exp92", "exp93", "exp94"],
            "transect": ["exp109", "exp114", "exp174"],
            "transect_random": ["exp139", "exp140", "exp141", "exp142", "exp143", "exp144", "exp145", "exp146",
                                "exp147", "exp148", "exp175", "exp176", "exp177", "exp178", "exp179"],
            "cylinder": ["exp104", "exp105", "exp106", "exp107", "exp108"],
            "cylinder_random": ["exp149", "exp150", "exp151", "exp152", "exp153",
                              "exp154", "exp155", "exp156", "exp157", "exp158",
                              "exp159", "exp160", "exp161", "exp162", "exp163",
                              "exp164", "exp165", "exp166", "exp167", "exp168",
                              "exp169", "exp170", "exp171", "exp172", "exp173"],
            "combined": ["exp180"],
            "per_feature": ["exp95", "exp96", "exp97", "exp98", "exp99", "exp100", "exp101", "exp102", "exp103"]
        }
    },
    "time_space_attention": {
        "masking": {
            "per_sample": ["exp256", "exp257", "exp258", "exp259", "exp260"],
            "transect": ["exp291", "exp297", "exp303"],
            "transect_random": ["exp292", "exp293", "exp294", "exp295", "exp296", "exp298", "exp299", "exp300",
                                "exp301", "exp302", "exp304", "exp305", "exp306", "exp307", "exp308"],
            "cylinder": ["exp261", "exp267", "exp273", "exp279", "exp285"],
            "cylinder_random": ["exp262", "exp263", "exp264", "exp265", "exp266",
                              "exp268", "exp269", "exp270", "exp271", "exp272",
                              "exp274", "exp275", "exp276", "exp277", "exp278",
                              "exp280", "exp281", "exp282", "exp283", "exp284",
                              "exp286", "exp287", "exp288", "exp289", "exp290"],
            "combined": ["exp318"],
            "per_feature": ["exp309", "exp310", "exp311", "exp312", "exp313", "exp314", "exp315", "exp316", "exp317"]
        }
    },
}


## Per-sample masking

Randomly hides N $\in$ {1, 2, 3, 4, 5} features per sample (mask ratios 0.3, 0.4, 0.5, 0.7, 0.9). Simulates missing-at-random (MAR) data gaps.

In [ ]:
def assemble_df(exp_ids):
    """Compiling relevant information of passed experiment IDs into a df (one row per experiment x seed)."""
    per_sample = []
    for exp_id in exp_ids:
        try:
            df_mask, _, _ = load_experiments([exp_id], attr_list=[])
        except ValueError:
            continue  # experiment not found locally - skip
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        # Attach masking config
        df_mask["cylinder_mask_radius"] = ablation_study[exp_id]["config"].cylinder_mask_radius
        df_mask["transect_mask_width"] = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)

        per_sample.append(df_mask)

    if not per_sample:
        print(f"No per-sample masking data available for {exp_ids}")
        return None

    # Translate mask_ratio to number of hidden features (for per-sample masking)
    df_per_sample = pd.concat(per_sample)
    df_per_sample["num_feat"] = df_per_sample["mask_ratio"].map(per_sample_num_map)
    df_per_sample["masking_strategy"] = "Per-sample"

    return df_per_sample


In [ ]:
# Plotting functions for different masking errors
def plot_masking_rmse(df_in, save_as=None, dpi=300):
    """RMSE vs. number of randomly hidden features per sample, split by attention type."""
    plt.figure(figsize=(6, 4))
    barplot_seeds(df_in, x="num_feat", y="RMSE", hue="Attention type")
    plt.xlabel("Number of randomly hidden features per sample")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()


def plot_masking_kge(df_in, save_as=None, dpi=300):
    """KGE vs. number of randomly hidden features per sample, split by attention type."""
    fig, ax = plt.subplots(figsize=(6, 4))
    barplot_seeds(df_in, x="num_feat", y="KGE", hue="Attention type", ax=ax)
    ax.set_xlabel("Number of randomly hidden features per sample")
    ax.set_ylabel("Log KGE" if safe_log_yscale(ax, df_in["KGE"]) else "KGE")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()


def plot_masking_woa_diff(df_in, save_as=None, dpi=300):
    """|WOA - pred| vs. number of hidden features, with the per-config WOA std as a second panel."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    temp = df_in.sort_values("num_feat").copy()
    temp["num_feat"] = temp["num_feat"].astype(str)

    # Bar plot of differences
    barplot_seeds(temp, x="num_feat", y="WOA_diff", hue="Attention type", ax=ax1)
    ax1.set_ylabel("Log |WOA - pred|" if safe_log_yscale(ax1, temp["WOA_diff"]) else "|WOA - pred|")

    # Add std across seeds
    sns.scatterplot(data=temp, x="num_feat", y="WOA_std", ax=ax2, hue="Attention type", legend=False)
    ax2.set_ylabel("Log std" if safe_log_yscale(ax2, temp["WOA_std"]) else "std")
    ax2.set_xlabel("Number of randomly hidden features per sample")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()


def plot_masking_cmip_diff(df_in, save_as=None, dpi=300):
    """|CMIP - pred| vs. number of hidden features, with the per-config CMIP std as a second panel."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    temp = df_in.sort_values("num_feat").copy()
    temp["num_feat"] = temp["num_feat"].astype(str)

    # Bar plot
    barplot_seeds(temp, x="num_feat", y="CMIP_diff", hue="Attention type", ax=ax1)
    ax1.set_ylabel("Log |CMIP - pred|" if safe_log_yscale(ax1, temp["CMIP_diff"]) else "|CMIP - pred|")

    # Add std across seeds
    sns.scatterplot(data=temp, x="num_feat", y="CMIP_std", ax=ax2, hue="Attention type", legend=False)
    ax2.set_ylabel("Log std" if safe_log_yscale(ax2, temp["CMIP_std"]) else "std")
    ax2.set_xlabel("Number of randomly hidden features per sample")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()


In [ ]:
# Get info for each attention model
df_per_sample = []
for attn_key, exp_dict in exp_map.items():
    exp_ids = exp_dict["masking"]["per_sample"]
    temp = assemble_df(exp_ids)
    if temp is None:
        continue
    temp["Attention type"] = attention_name_map[attn_key]
    df_per_sample.append(temp)
df_per_sample = pd.concat(df_per_sample) if df_per_sample else pd.DataFrame()

# Error plots
if df_per_sample.empty:
    print("No per-sample masking data available locally.")
else:
    plot_masking_rmse(df_per_sample, save_as="output/plots/ablation/masking/masking_rmse.png", dpi=300)
    plot_masking_kge(df_per_sample, save_as="output/plots/ablation/masking/masking_kge.png", dpi=300)
    plot_masking_cmip_diff(df_per_sample, save_as="output/plots/ablation/masking/masking_cmip_diff.png", dpi=300)
    plot_masking_woa_diff(df_per_sample, save_as="output/plots/ablation/masking/masking_woa_diff.png", dpi=300)


## Transect masking

Masks all features along a vertical or horizontal transect of fixed width. Simulates ship-track or mooring-line gaps of width `transect_mask_width` and orientation 0=random, 1=axis-aligned.

In [ ]:
def assemble_transect_df(exp_ids_transect, exp_ids_transect_random):
    """Compiling relevant information of passed experiment IDs into a df (one row per experiment x seed)."""
    transect = []
    for exp_id in exp_ids_transect + exp_ids_transect_random:
        try:
            df_mask, _, _ = load_experiments([exp_id], attr_list=[])
        except ValueError:
            continue  # experiment not found locally - skip
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        # Get transect mask width
        transect_width = ablation_study[exp_id]["config"].transect_mask_width

        # Attach masking config
        df_mask["transect_mask_width"] = transect_width
        df_mask["cylinder_mask_radius"] = ablation_study[exp_id]["config"].cylinder_mask_radius
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)

        # Orientation: 0=fully random orientation, 1=axis-aligned only
        df_mask["masking_strategy"] = (
            f"Transect (w={transect_width})" if df_mask["transect_mask_orientation"].values[0] == 0
            else f"Transect (w={transect_width}, aligned)"
        )

        transect.append(df_mask)

    if not transect:
        print(f"No transect masking data available for {exp_ids_transect + exp_ids_transect_random}")
        return None

    # Translate mask_ratio to number of hidden features (for per-sample masking)
    df_transect = pd.concat(transect)
    df_transect["num_feat"] = df_transect["mask_ratio"].map(per_sample_num_map)
    df_transect.loc[df_transect["exp_id"].isin(exp_ids_transect), "num_feat"] = 0

    return df_transect


In [ ]:
# Get transect masking results
df_transect = []
for attn_key, exp_dict in exp_map.items():
    exp_ids_transect = exp_dict["masking"]["transect"]
    exp_ids_transect_random = exp_dict["masking"]["transect_random"]
    temp = assemble_transect_df(exp_ids_transect, exp_ids_transect_random)
    if temp is None:
        continue
    temp["Attention type"] = attention_name_map[attn_key]
    df_transect.append(temp)
df_transect = pd.concat(df_transect) if df_transect else pd.DataFrame()


## Cylindrical masking

Masks all features within a spatial cylinder of radius `cylinder_mask_radius` (degrees). Simulates regional data gaps such as under-sampled ocean basins.

In [ ]:
def assemble_cylinder_df(exp_ids_cylinder, exp_ids_cylinder_random):
    """Compiling relevant information of passed experiment IDs into a df (one row per experiment x seed)."""
    cylinder = []
    for exp_id in exp_ids_cylinder + exp_ids_cylinder_random:
        try:
            df_mask, _, _ = load_experiments([exp_id], attr_list=[])
        except ValueError:
            continue  # experiment not found locally - skip
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        # Attach masking config
        df_mask["transect_mask_width"] = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["cylinder_mask_radius"] = ablation_study[exp_id]["config"].cylinder_mask_radius
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)

        # Geometrically a cylinder (circle in lat/lon, unrestricted in depth/time)
        df_mask["masking_strategy"] = "Cylinder (r=" + str(df_mask["cylinder_mask_radius"].values[0]) + ")"
        cylinder.append(df_mask)

    if not cylinder:
        print(f"No cylindrical masking data available for {exp_ids_cylinder + exp_ids_cylinder_random}")
        return None

    # Translate mask_ratio to number of hidden features (for per-sample masking)
    df_cylindrical = pd.concat(cylinder)
    df_cylindrical["num_feat"] = df_cylindrical["mask_ratio"].map(per_sample_num_map)
    df_cylindrical.loc[df_cylindrical["exp_id"].isin(exp_ids_cylinder), "num_feat"] = 0

    return df_cylindrical


In [ ]:
# Get cylindrical masking results
df_cylinder = []
for attn_key, exp_dict in exp_map.items():
    exp_ids_cylinder = exp_dict["masking"]["cylinder"]
    exp_ids_cylinder_random = exp_dict["masking"]["cylinder_random"]
    temp = assemble_cylinder_df(exp_ids_cylinder, exp_ids_cylinder_random)
    if temp is None:
        continue
    temp["Attention type"] = attention_name_map[attn_key]
    df_cylinder.append(temp)
df_cylinder = pd.concat(df_cylinder) if df_cylinder else pd.DataFrame()


## Combined

Applies a mix of per-sample, transect, and cylindrical masking within a single training run to expose the model to all gap geometries simultaneously.

In [ ]:
def assemble_combined_df(exp_ids):
    """Compiling relevant information of passed experiment IDs into a df (one row per experiment x seed)."""
    combined = []
    for exp_id in exp_ids:
        try:
            df_mask, _, _ = load_experiments([exp_id], attr_list=[])
        except ValueError:
            continue  # experiment not found locally - skip
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)

        # Get relevant settings
        df_mask["transect_mask_width"] = ablation_study[exp_id]["config"].transect_mask_width
        df_mask["cylinder_mask_radius"] = ablation_study[exp_id]["config"].cylinder_mask_radius
        df_mask["masking_strategies"] = str(ablation_study[exp_id]["config"].masking_strategies)
        df_mask["masking_strategy"] = "Combined"
        combined.append(df_mask)

    if not combined:
        print(f"No combined masking data available for {exp_ids}")
        return None

    # Translate mask_ratio to number of hidden features (for per-sample masking)
    df_combined = pd.concat(combined)
    df_combined["num_feat"] = df_combined["mask_ratio"].map(per_sample_num_map)
    df_combined.loc[df_combined["exp_id"].isin(exp_ids), "num_feat"] = 1

    return df_combined


In [ ]:
# Get results for combined masking
df_combined = []
for attn_key, exp_dict in exp_map.items():
    exp_ids_combined = exp_dict["masking"]["combined"]
    temp = assemble_combined_df(exp_ids_combined)
    if temp is None:
        continue
    temp["Attention type"] = attention_name_map[attn_key]
    df_combined.append(temp)
df_combined = pd.concat(df_combined) if df_combined else pd.DataFrame()


## Heatmaps

Summarises RMSE, CMIP diff, and WOA diff by masking strategy and number of hidden features.

In [ ]:
def _relative_luminance(rgba):
    """Web Content Accessibility Guidelines (WCAG) relative luminance of an RGB(A) colour (0 = black, 1 = white)."""
    c = np.array(rgba[:3], dtype=float)
    c = np.where(c <= 0.03928, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    return 0.2126 * c[0] + 0.7152 * c[1] + 0.0722 * c[2]


def _draw_heatmap_annotations(ax, value_pivot, cv_pivot, decimals, cmap_name, fontsize=None):
    """Draw per-cell hatch (seed-instability of this metric) + halo around value text."""
    import matplotlib.patheffects as pe

    HATCH_LEVELS = ["", "/", "//", "///", "////"]  # five hatch levels

    # Define norm for coefficient of variation (cv)
    finite_cv = cv_pivot.to_numpy()[np.isfinite(cv_pivot.to_numpy())]
    if finite_cv.size and finite_cv.max() > finite_cv.min():
        cv_norm = (cv_pivot - finite_cv.min()) / (finite_cv.max() - finite_cv.min())
    else:
        cv_norm = cv_pivot * 0  # no relative signal to show - treat all as equally confident

    def cv_to_hatch(v):
        # Map coefficient of variation to hatch level
        if pd.isna(v):
            return HATCH_LEVELS[-1]
        idx = int(np.clip(v, 0, 1) * (len(HATCH_LEVELS) - 1) + 0.5)
        return HATCH_LEVELS[idx]

    # Define matplotlib norm and cmap
    norm = mcolors.Normalize(vmin=np.nanmin(value_pivot.values), vmax=np.nanmax(value_pivot.values))
    cmap = plt.get_cmap(cmap_name)
    halo = 2.5 if fontsize is None else 0.25 * fontsize

    # Iterate cell-by-cell (row=y=masking_strategy, col=x=num_feat)
    for y in range(value_pivot.shape[0]):
        for x in range(value_pivot.shape[1]):
            val = value_pivot.iloc[y, x]
            if pd.isna(val):
                continue

            # Recover this cell's actual background colour (sns.heatmap does not expose it)
            rgba = cmap(norm(val))

            # Contrast decides whether text/hatch reads better in black or white (always >= 4.5:1)
            text_color = "black" if _relative_luminance(rgba) > 0.179 else "white"
            text = f"{val:.{decimals}f}"

            # Hatch density encodes seed-to-seed instability (this cell's CV)
            hatch_pattern = cv_to_hatch(cv_norm.iloc[y, x])
            if hatch_pattern:
                ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor="none", hatch=hatch_pattern,
                                           edgecolor=text_color, linewidth=0, alpha=0.5, zorder=2))

            # Halo keeps the digits readable
            ax.text(x + 0.5, y + 0.5, text, ha="center", va="center", color=text_color, zorder=3,
                    fontsize=fontsize, path_effects=[pe.withStroke(linewidth=halo, foreground=rgba)])


def _finish_heatmap(ax, title, fontsize, show_ylabels):
    """Titles, fonts, no frames, optionally hide the row labels."""
    ax.set_title(title, fontsize=fontsize)  # title

    # Labels
    ax.set_xlabel("Num. features masked", fontsize=fontsize)
    ax.set_ylabel("Masking strategy" if show_ylabels else "", fontsize=fontsize)

    # Ticks
    if fontsize is not None:
        ax.tick_params(labelsize=fontsize, length=0)
    if not show_ylabels:
        ax.set_yticklabels([])

    sns.despine(ax=ax, left=True, bottom=True)  # no top and right spines
    ax.grid(False)  # grid

    # Colourbar
    cbar = ax.collections[0].colorbar
    cbar.outline.set_visible(False)
    if fontsize is not None:
        cbar.ax.tick_params(labelsize=fontsize, length=2, width=0.5)

    # Hatch settings
    plt.rcParams["hatch.linewidth"] = 0.4 if fontsize is not None else 1.0


# Human-readable heatmap titles for metrics whose column name isn't presentation-ready
HEATMAP_TITLES = {"CMIP_diff": "CMIP6 MAE", "WOA_diff": "WOA MAE"}


def plot_masking_heatmap(df, error_name="RMSE", save_as=None, dpi=150, figsize=(8, 5), fontsize=None, annot_fontsize=None, show_ylabels=True, label_width=None):
    """Plot heatmap for various masking strategies with varying parameters.

    Hatch density encodes this metric's coefficient of variation across the 5 seeds
    (std/mean, min-max normalised within the heatmap). Not enough seeds for a proper
    significance test.
    """
    # Get masking strategies
    strategies = sorted(df["masking_strategy"].unique())
    strategies = ["Per-sample"] + [s for s in strategies if s not in ["Per-sample", "Combined"]] + ["Combined"]

    # Pivot error into heatmap format
    error_pivot = df.pivot_table(
        index="masking_strategy", columns="num_feat", values=error_name, aggfunc="mean"
    ).reindex(strategies)

    if error_pivot.empty or error_pivot.shape[1] == 0:
        print(f"No {error_name} data available - skipping heatmap")
        return

    # CV computed before any display-unit conversion below
    std_pivot = df.pivot_table(
        index="masking_strategy", columns="num_feat", values=error_name, aggfunc="std"
    ).reindex(strategies).reindex(columns=error_pivot.columns)
    cv_pivot = std_pivot / error_pivot

    # Decimal precision per metric
    HEATMAP_DECIMALS = {"RMSE": 3, "KGE": 2, "CMIP_diff": 3, "WOA_diff": 3}
    display_name = HEATMAP_TITLES.get(error_name, error_name)
    decimals = HEATMAP_DECIMALS.get(error_name, 3)

    # Darker = better for every metric
    cmap_name = "magma_r" if error_name == "KGE" else "magma"
    if error_name == "Total time":
        # Total time is more readable in hours than in raw seconds at these magnitudes
        error_pivot = error_pivot / 3600
        display_name = "Total time [h]"
        decimals = 1

    # Wrap long row labels
    if label_width:
        import textwrap
        error_pivot.index = [textwrap.fill(str(s), label_width) for s in error_pivot.index]

    # Plot heatmap
    fig, ax1 = plt.subplots(figsize=figsize, layout="constrained")
    sns.heatmap(error_pivot, annot=False, cmap=cmap_name, ax=ax1, cbar_kws={"aspect": 20}, yticklabels=True, xticklabels=True)
    _finish_heatmap(ax1, display_name, fontsize, show_ylabels)

    # Add annotations
    _draw_heatmap_annotations(ax1, error_pivot, cv_pivot, decimals=decimals, cmap_name=cmap_name, fontsize=annot_fontsize)

    if save_as:
        plt.savefig(save_as, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    plt.show()


def plot_masking_miss_ratio(df, save_as=None, dpi=150, figsize=(8, 5), fontsize=None, annot_fontsize=None, show_ylabels=True):
    """Plot the effective missing-value fraction by masking strategy and num_feat."""
    # Get masking strategies
    strategies = sorted(df["masking_strategy"].unique())
    strategies = ["Per-sample"] + [s for s in strategies if s not in ["Per-sample", "Combined"]] + ["Combined"]

    # Check if the ratio-column is given
    if "effective_miss_ratio" not in df.columns or not df["effective_miss_ratio"].notna().any():
        print("No effective_miss_ratio data available - skipping")
        return

    # Pivot missingness information (and std) into heatmap format
    miss_pivot = df.pivot_table(
        index="masking_strategy", columns="num_feat", values="effective_miss_ratio", aggfunc="mean"
    ).reindex(strategies)
    std_pivot = df.pivot_table(
        index="masking_strategy", columns="num_feat", values="effective_miss_ratio", aggfunc="std"
    ).reindex(strategies).reindex(columns=miss_pivot.columns)
    cv_pivot = std_pivot / miss_pivot

    # Plot heatmap
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")
    sns.heatmap(miss_pivot, annot=False, cmap="magma", ax=ax, cbar_kws={"aspect": 20}, yticklabels=True, xticklabels=True)
    _finish_heatmap(ax, "Fraction of hidden values", fontsize, show_ylabels)

    # Add annotations
    _draw_heatmap_annotations(ax, miss_pivot, cv_pivot, decimals=3, cmap_name="magma", fontsize=annot_fontsize)

    if save_as:
        plt.savefig(save_as, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    plt.show()


def plot_masking_pair(df, error_name="RMSE", right_name="effective_miss_ratio", save_as=None, dpi=150, width=5.67,
                      height=2.6, fontsize=7, annot_fontsize=5.5, label_width=18):
    """Plot 2 heatmaps side-by-side as one figure.

    Parameters
    ----------
    width, height : float
        Figure size in inches.
    fontsize, annot_fontsize : float
        Font size of titles/labels/colourbars and of the numbers inside the cells.
    label_width : int, optional
        Maximum characters per line of the row labels (long labels are wrapped).
    """
    import textwrap

    # Get masking strategies
    strategies = sorted(df["masking_strategy"].unique())
    strategies = ["Per-sample"] + [s for s in strategies if s not in ["Per-sample", "Combined"]] + ["Combined"]

    def _pivots(col):
        mean = df.pivot_table(index="masking_strategy", columns="num_feat", values=col, aggfunc="mean").reindex(strategies)
        std = df.pivot_table(index="masking_strategy", columns="num_feat", values=col, aggfunc="std"
                             ).reindex(strategies).reindex(columns=mean.columns)
        return mean, std / mean

    # Pivot input data
    err_pivot, err_cv = _pivots(error_name)
    miss_pivot, miss_cv = _pivots(right_name)

    # Wrap long row labels so that the cells can be wider
    if label_width:
        wrapped = [textwrap.fill(str(s), label_width) for s in strategies]
        err_pivot.index, miss_pivot.index = wrapped, wrapped

    # Colour
    cmap_name = "magma_r" if error_name == "KGE" else "magma"
    right_cmap = "magma_r" if right_name == "KGE" else "magma"

    # Titles
    display_name = HEATMAP_TITLES.get(error_name, error_name)  # title
    right_title = "Fraction of hidden values" if right_name == "effective_miss_ratio" else HEATMAP_TITLES.get(right_name, right_name)

    # KGE is shown with 2 decimals, all other metrics with 3
    left_dec, right_dec = (2 if error_name == "KGE" else 3), (2 if right_name == "KGE" else 3)

    # Heatmap plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width, height), layout="constrained")
    for ax, pivot in ((ax1, err_pivot), (ax2, miss_pivot)):
        sns.heatmap(pivot, annot=False, cmap=cmap_name if ax is ax1 else right_cmap, ax=ax,
                    cbar_kws={"aspect": 20}, yticklabels=True, xticklabels=True)

    # Polishing
    _finish_heatmap(ax1, display_name, fontsize, True)
    _finish_heatmap(ax2, right_title, fontsize, False)

    # Add annotations
    _draw_heatmap_annotations(ax1, err_pivot, err_cv, decimals=left_dec, cmap_name=cmap_name, fontsize=annot_fontsize)
    _draw_heatmap_annotations(ax2, miss_pivot, miss_cv, decimals=right_dec, cmap_name=right_cmap, fontsize=annot_fontsize)

    # Save
    if save_as:
        plt.savefig(save_as, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    plt.show()

In [ ]:
df_all = pd.concat([df_per_sample, df_transect, df_cylinder, df_combined], ignore_index=True)
df_all["Total time"] = df_all["train_time"] + df_all["pred_time"]

# Heatmap plot per model and error type
for attn_key, exp_dict in exp_map.items():
    attn_label = attention_name_map[attn_key]
    if "Attention type" not in df_all.columns or attn_label not in df_all["Attention type"].unique():
        print(f"No masking data available for {attn_label} - skipping heatmap")
        continue
    print(attn_label)

    temp = df_all[df_all["Attention type"] == attn_label]

    # Plot RMSE and fraction of hidden values
    plot_masking_pair(temp, "RMSE", right_name="effective_miss_ratio",
                      save_as=f"output/plots/ablation/masking/masking_heatmap_{attn_label}_RMSE.pdf", dpi=1000)

    # Supplement figure (final width = A4 text width): KGE (left) and CMIP6 MAE (right) in one figure
    plot_masking_pair(temp, "KGE", right_name="CMIP_diff",
                      save_as=f"output/plots/ablation/masking/masking_heatmap_{attn_label}_KGE_CMIP_diff.pdf", dpi=1000,
                      width=6.69, height=2.8)

    # One heatmap per remaining metric
    for error_name in ["WOA_diff", "Total time"]:
        plot_masking_heatmap(
            df=temp,
            error_name=error_name,
            save_as=f"output/plots/ablation/masking/masking_heatmap_{attn_label}_{error_name}.pdf",
            dpi=1000,
            figsize=(3.55, 2.6),
            fontsize=7,
            annot_fontsize=6,
            label_width=18
        )


In [ ]:
# Metric correlations
for attn_label in df_all["Attention type"].unique():
    print(attn_label)
    temp = df_all[df_all["Attention type"] == attn_label]
    temp_corr = temp[["RMSE", "WOA_diff", "CMIP_diff", "effective_miss_ratio", "Total time"]].corr()
    print(temp_corr)
    print()


In [ ]:
# Visual map comparison
MAP_COMPARISON = [
    ("exp245", "No feature mixer, mask ratio 0.3"),
    ("exp256", "Feature mixer, mask ratio 0.3"),
    ("exp269", "Feature mixer, best masking (2 features per sample + cylinder, r=0.3)"),
]

# Data loading
pred_data_raw = {}
col_items_smooth = []
for exp_id, label in MAP_COMPARISON:
    key = f"{model_name}_{exp_id}"
    col_items_smooth.append((key, textwrap.fill(label, width=28)))

    file_name = ablation_base_dir / exp_id
    df_imputed, n_seeds_used = load_averaged_predictions(file_name, model_name)
    print(f"{label} ({exp_id}): {n_seeds_used} seed(s)")
    pred_data_raw[key] = df_imputed

# Get first and last year
dates = sorted(pred_data_raw[f"{model_name}_exp245"]["DATEANDTIME"].unique())
times = [dates[0], dates[-1]]
print(f"Using times={times}")

# Nearest-to-surface depth per experiment
pred_data_smooth = {}
for key, df in pred_data_raw.items():
    nearest_depth = min(df["LEV_M"].unique(), key=lambda d: abs(d))
    pred_data_smooth[key] = df[(df["LEV_M"] == nearest_depth) & (df["DATEANDTIME"].isin(times))]
    print(f"{key}: depth={nearest_depth}, {len(pred_data_smooth[key])} rows after filtering")

# One map plot per parameter
for parameter in config.parameters:
    gridded = {key: df_to_gridded_da(pred_data_smooth[key], value_col=parameter) for key, _ in col_items_smooth}
    plot_geo_panels(
        get_data=lambda time, m, _g=gridded: _g[m].sel(time=int(time), depth=float(pred_data_smooth[m]["LEV_M"].iloc[0])),
        row_items=[(t, str(int(t))) for t in times],
        col_items=col_items_smooth,
        colorbar_label=config.parameter_name_unit_map.get(parameter, parameter),
        plot_config={"cmap": "inferno"},
        output_path="output/plots/ablation/masking",
        filename=f"feature_mixer_progression_{parameter}.pdf",
        dpi=1000,
        width=6.69,  # supplement text width, fonts are real print sizes
        fontsize=8,
        tick_fontsize=6,
        nodate_colour="0.55",
    )


## Per-feature masking

Masks a fixed fraction of all samples per feature.

In [ ]:
def assemble_per_feature_df(exp_ids):
    """Compiling relevant information of passed experiment IDs into a df (one row per experiment x seed)."""
    per_feature = []
    for exp_id in exp_ids:
        try:
            df_mask, _, _ = load_experiments([exp_id], attr_list=[])
        except ValueError:
            continue  # experiment not found locally - skip

        # Get fraction of hidden values
        df_mask["effective_miss_ratio"] = extract_mean_miss_ratio(exp_id)
        per_feature.append(df_mask)

    if not per_feature:
        print(f"No per-feature masking data available for {exp_ids}")
        return None
    return pd.concat(per_feature)


def plot_masking_per_feature_error(df_in, x="mask_ratio", y="RMSE", y_label=None,
                                   y_scale=None, save_as=None, dpi=300):
    """Error bar plot for per-feature masking."""
    # Bar plot
    fig, ax = plt.subplots(figsize=(6, 4))
    barplot_seeds(df_in, x=x, y=y, hue="Attention type", ax=ax)
    ax.set_xlabel("Mask ratio")
    label = y_label

    # Apply log-scale if possible
    if y_scale == "log":
        applied = safe_log_yscale(ax, df_in[y])
        if not applied and label and label.startswith("Log "):
            label = label[len("Log "):]  # log scale was not applicable

    # Update y-label
    if label:
        ax.set_ylabel(label)

    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=dpi)
    plt.show()


In [ ]:
# Get results for per-feature masking
df_per_feature = []
for attn_key, exp_dict in exp_map.items():
    exp_ids = exp_dict["masking"]["per_feature"]
    temp = assemble_per_feature_df(exp_ids)
    if temp is None:
        continue
    temp["Attention type"] = attention_name_map[attn_key]
    df_per_feature.append(temp)
df_per_feature = pd.concat(df_per_feature, ignore_index=True) if df_per_feature else pd.DataFrame()

# Plotting
if df_per_feature.empty:
    print("No per-feature masking data available locally.")
else:
    plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="RMSE",
                                   save_as="output/plots/ablation/masking/masking_per_feature_rmse.png", dpi=300)
    plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="CMIP_diff",
                                   y_label="Log |CMIP - pred|", y_scale="log",
                                   save_as="output/plots/ablation/masking/masking_per_feature_CMIP_diff.png", dpi=300)
    plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="WOA_diff",
                                   y_label="|WOA - pred|",
                                   save_as="output/plots/ablation/masking/masking_per_feature_WOA_diff.png", dpi=300)
    plot_masking_per_feature_error(df_per_feature, x="mask_ratio", y="KGE",
                                   y_label="KGE", y_scale="log",
                                   save_as="output/plots/ablation/masking/masking_per_feature_KGE.png", dpi=300)


In [ ]:
# Metric correlations
for attn_label in df_per_feature["Attention type"].unique():
    print(attn_label)
    temp = df_per_feature[df_per_feature["Attention type"] == attn_label]
    temp_corr = temp[["RMSE", "WOA_diff", "CMIP_diff", "effective_miss_ratio"]].corr()
    print(temp_corr)
    print()


In [ ]:
# Save results on disc
df_all.to_csv("masking_ablation.csv", index=False)

In [ ]:
# Latex table for per-feature masking: One row per mask ratio
t = df_per_feature[df_per_feature["Attention type"] == "Time-space attention"]
t = aggregate_seeds(t, group_cols=["mask_ratio"])
warn_incomplete_seeds(t, id_col="mask_ratio")
t = t.rename(columns={"mask_ratio": "Mask ratio", "effective_miss_ratio": "Effective missingness [%]"})
t["Effective missingness [%]"] = round((t["Effective missingness [%]"] * 100))
print(generate_latex_table(
    df_global=t, df_pervar=pd.DataFrame(), model_col="Mask ratio", include_pervar=False,
    bold_best=True,
    caption="RMSE and KGE per mask ratio (Time-space attention, seed-averaged).",
    label="tab:sm:mask_ratio_per_feature",
))

# Overview: all experiments

Flat directory scan loads global and per-variable metrics for all architecture ablation experiments , then ranks them. Masking-strategy experiments are excluded from the ranking since they vary training distribution rather than architecture.

In [ ]:
model_name = "sonati"

# Load metrics by scanning ablation directories (one row per experiment x seed)
global_rows = []
pervar_rows = []
base_path = Path(config.output_dir_tuning) / model_name / "architecture_ablation"
# base_path2 = Path("D:/phd/output/tuning/sonati/architecture_ablation/")
base_path2 = Path("F:/PhD/phd_repos/time_imputation/output/tuning/sonati/architecture_ablation/")

seen = set()  # deduplicate paths that exist in both base dirs
for out_dir in glob(str(base_path / "*")) + glob(str(base_path2 / "*")):
    out_dir = Path(out_dir)
    if out_dir.name in seen:
        continue
    seen.add(out_dir.name)

    # Iterate over file names
    exp_id = out_dir.name
    for file_path in find_result_files(out_dir, model_name):
        res = json.load(open(file_path, "r"))  # load results
        metrics_key = "test_metrics" if res["model_framework"] == "pytorch" else "metrics_all"
        pervar_dict = res[metrics_key]["PerVariable"]

        # Assemble architecture and metric information
        global_entry = {
            "exp_id": exp_id,
            "seed": res.get("seed", 0),
            "attention_type": res["hyps"]["model"]["cfg"].get("attention_type", ""),
            **res[metrics_key]["Global"]
        }

        # Take median of KGE (not mean)
        kge_pervar = [v["KGE"] for v in pervar_dict.values() if not np.isnan(v.get("KGE", np.nan))]
        if kge_pervar:
            global_entry["KGE"] = np.median(kge_pervar)
        global_rows.append(global_entry)

        for param, metrics in pervar_dict.items():
            pervar_rows.append({
                "exp_id": exp_id,
                "seed": res.get("seed", 0),
                "parameter": param,
                **metrics
            })

df_global_seeds = pd.DataFrame(global_rows)
df_pervar = pd.DataFrame(pervar_rows)
df_pervar_long = df_pervar.melt(id_vars=["exp_id", "seed", "parameter"], var_name="metric", value_name="value")

# One row per experiment (mean over seeds) - used for ranking/plotting below
df_global = aggregate_seeds(df_global_seeds, group_cols=["exp_id"])


In [ ]:
# Exclude masking experiments
df_global = df_global[~df_global.exp_id.isin(masking_exp_ids)]
df_global_seeds = df_global_seeds[~df_global_seeds.exp_id.isin(masking_exp_ids)]
df_pervar = df_pervar[~df_pervar.exp_id.isin(masking_exp_ids)]
warn_incomplete_seeds(df_global, id_col="exp_id")


In [ ]:
# Global ranking across all metrics
df = df_global.copy()

ranked = pd.DataFrame()
ranked["exp_id"] = df["exp_id"]

for min_metric in ["MAE", "RMSE"]:
    if min_metric in df.columns:
        ranked[min_metric] = df[min_metric].rank(ascending=True)

for max_metric in ["R2", "KGE"]:
    if max_metric in df.columns:
        ranked[max_metric] = df[max_metric].rank(ascending=False)

ranked["total_rank"] = ranked.drop(columns=["exp_id"]).mean(axis=1)
df_ranked = ranked.sort_values("total_rank")
df_ranked.head(5)


In [ ]:
# Plot (per-seed spread, ordered by mean RMSE)
error_name = "RMSE"
temp = df_global_seeds.sort_values(error_name, ascending=False)
exp_order = df_global.sort_values(error_name, ascending=False)["exp_id"].tolist()

plt.figure(figsize=(12, 5))
barplot_seeds(temp, x="exp_id", y=error_name, hue="attention_type", order=exp_order)
plt.xticks(rotation=90)
plt.xlabel("")
plt.tight_layout()
# plt.savefig("output/plots/sonati_ablation_rmse.png")
plt.show()

# RMSE vs R2 trade-off (mean per experiment)
error_name = "RMSE"
plt.figure(figsize=(20, 8))
plt.scatter(df_global[error_name], df_global["R2"])
for i, row in df_global.iterrows():
    plt.text(row[error_name], row["R2"], row["exp_id"])
plt.xlabel(f"{error_name} (lower better)")
plt.ylabel("R2 (higher better)")
plt.title("Trade-off: Error vs Explained Variance")
plt.show()


In [ ]:
# Ranking of time-space attention experiments
min_metrics_ov = ["RMSE", "WOA_diff", "CMIP_diff"]

df_tsa = df_global[df_global["attention_type"] == "time_space_attention"].copy()

ranked_tsa = pd.DataFrame()
ranked_tsa["exp_id"] = df_tsa["exp_id"]

for min_metric in min_metrics_ov:
    if min_metric in df_tsa.columns:
        ranked_tsa[min_metric] = df_tsa[min_metric].rank(ascending=True)

ranked_tsa["total_rank"] = ranked_tsa.drop(columns=["exp_id"]).mean(axis=1)
ranked_tsa.sort_values("total_rank").head(5)


# Graph construction

Ablates the k-NN graph used to select each query point's neighbourhood. Covers: static vs. dynamic graph construction, learnable anisotropic edge weights, input encoding for the graph encoder, and update/freeze schedules for dynamic graphs.

In [ ]:
# KNN graph ablation
attr_list = ["graph_mode", "graph_space", "graph_metric", "graph_update_frequency",
             "graph_warmup", "graph_freeze_epoch", "learn_anisotropic_weights",
             "encoder_scope", "encoder_input"]
error_name = "RMSE"

id_desc_map = {
    0: "Static",
    9: "Feature-difference-weighted distances",
    43: "Dynamic (learnable anisotropic weights, softplus)",
    45: "Dynamic (learnable anisotropic weights, exp)",
    40: "Dynamic",
    35: "No warmup, update every epoch, freeze after 20 epochs",
    34: "No warmup, update every 10 epochs, freeze after 50 epochs",
    41: "Warmup 20 epochs, update every epoch, no freeze",
    36: "Warmup 20 epochs, update every 5 epochs, no freeze",
}


def _to_desc(exp_id_col):
    """Map "expN" exp_id strings to their human-readable description via id_desc_map."""
    return exp_id_col.str.replace("exp", "", regex=False).astype(int).map(id_desc_map)


# Basic KNN setups
try:
    r1, df_woa, df_cmip = load_experiments([0, 9, 40, 45, 43], attr_list, woa=True)
    t1 = r1.sort_values(error_name, ascending=False)
    t1["desc"] = _to_desc(t1["exp_id"])

    plt.figure(figsize=(10, 6))
    barplot_seeds(t1, x=None, y=error_name, hue="desc")
    plt.xlabel("")
    plt.xticks(rotation=90)
    plt.title("Graph setups")
    plt.tight_layout()
    plt.savefig(f"{config.output_dir_plots}/ablation/sonati_ablation_knn_graph_{error_name}.png")
    plt.show()
except ValueError as e:
    print(f"Skipping basic KNN setups: {e}")

# Input encoding
try:
    r2, df_woa, df_cmip = load_experiments([12, 13, 14], attr_list, woa=True)
    t2 = r2.sort_values(error_name, ascending=False)
    t2["Encoder input"] = t2["encoder_input"].map({
        "coords_feat": "Coordinates, features",
        "coords_feat_mask": "Coordinates, features, masks",
        "coords": "Coordinates"
    })

    barplot_seeds(t2, x=None, y=error_name, hue="Encoder input")
    plt.xlabel("")
    plt.title("Input encodings (update every 5 epochs, freeze after 20 epochs)")
    plt.tight_layout()
    plt.savefig(f"{config.output_dir_plots}/ablation/sonati_ablation_knn_graph_input_encodings_{error_name}.png")
    plt.show()
except ValueError as e:
    print(f"Skipping input encoding: {e}")

# Update strategies
try:
    r3, df_woa, df_cmip = load_experiments([34, 35, 36, 41], attr_list, woa=True)
    t3 = r3.sort_values(error_name, ascending=False)
    t3["desc"] = _to_desc(t3["exp_id"])

    barplot_seeds(t3, x=None, y=error_name, hue="desc")
    plt.xlabel("")
    plt.title("Update strategies")
    plt.legend(loc="lower center")
    plt.tight_layout()
    plt.savefig(f"{config.output_dir_plots}/ablation/sonati_ablation_knn_graph_update_strategies_{error_name}.png")
    plt.show()
except ValueError as e:
    print(f"Skipping update strategies: {e}")


# Neighbourhood and positional encoding

Ablates neighbourhood size (k), the use of relative position vectors in attention (i.e. whether the model sees lat, lon, depth, time to each neighbour), and strategies for encoding absolute coordinates.

In [ ]:
# Number of neighbours ablation
exp_ids = [0, 19, 20, 21, 22, 23]
attr_list = ["n_neighbours"]
error_name = "RMSE"

results, df_woa, df_cmip = load_experiments(exp_ids, attr_list, woa=True)
results = results.sort_values(error_name, ascending=False)

sns.lineplot(results, x="n_neighbours", y=error_name, marker="o", errorbar="sd")
plt.xlabel("Number of neighbours")
plt.tight_layout()
plt.savefig(f"output/plots/ablation/sonati_ablation_n_neighbours_{error_name}.png")
plt.show()


In [ ]:
# Relative position encoding ablation
exp_ids = [0, 2, 15, 26, 3, 11, 27, 29, 38, 72, 78, 79, 75, 74, 81, 17, 31, 71, 82]
attr_list = ["attention_type", "feature_mixer_input", "feature_mixer", "use_masks",
             "use_rel_pos", "positional_encoding"]
error_name = "RMSE"

results, df_woa, df_cmip = load_experiments(exp_ids, attr_list, woa=True)
results = results.sort_values("attention_type")
results["model_label"] = (
        results["attention_type"] +
        np.where(results["feature_mixer"],
                 ", feature mixer (" + results["feature_mixer_input"].str.replace("_", "&") + ")", "") +
        np.where(results["use_masks"], ", attention masks", "")
)
results = results.rename(columns={"use_rel_pos": "Use relative positions"})

plt.figure(figsize=(6, 7))
barplot_seeds(results, x="model_label", y=error_name, hue="Use relative positions")
plt.xlabel("")
plt.xticks(rotation=90)
plt.title("Influence of relative position on various architectures")
plt.tight_layout()
plt.savefig(f"output/plots/ablation/sonati_ablation_rel_pos_{error_name}.png")
plt.show()


| exp_id | Architecture | Feature mixer | Masks | Rel pos |
|--------|--------------|---------------|-------|---------|
| 0  | MHA          |               |       |         |
| 2  | MHA          |               |       | rel_pos |
| 24 | MHA          |               |       | rel_pos (encoded) |
| 26 | MHA          | feat+mask     |       | rel_pos |
| 15 | MHA          | feat+mask     |       |         |
| 78 | MHA          | feat          |       | rel_pos |
| 3  | MHA          | feat          |       |         |
| 79 | Autoencoder  |               |       | rel_pos |
| 11 | Autoencoder  |               |       |         |
| 29 | Autoencoder  | feat+mask     |       | rel_pos |
| 27 | Autoencoder  | feat+mask     |       |         |
| 71 | Autoencoder  | feat          | masks | rel_pos |
| 38 | Autoencoder  | feat          | masks |         |
| 75 | Space-time   | feat          |       | rel_pos |
| 72 | Space-time   | feat          |       |         |

In [ ]:
# Coordinate encoding ablation
exp_ids = [0, 4, 5, 7, 8, 12, 13, 14]
attr_list = ["encoder_input", "encoder_scope", "graph_mode"]
error_name = "RMSE"

results, df_woa, df_cmip = load_experiments(exp_ids, attr_list, woa=True)
results = results.sort_values(error_name, ascending=False)

desc_map = {
    0: "No coord encoding",
    4: "Coord encoder (model) on coords",
    5: "Coord encoder (graph) on coords",
    7: "Coord encoder (both) on coords+feat",
    8: "Coord encoder (both) on coords+feat+mask",
    12: "Coord encoder (both) on coords (dynamic)",
    13: "Coord encoder (both) on coords+feat (dynamic)",
    14: "Coord encoder (both) on coords+feat+mask (dynamic)",
}
results["desc"] = results["exp_id"].str.replace("exp", "", regex=False).astype(int).map(desc_map)

plt.figure(figsize=(8, 5))
barplot_seeds(results, hue="desc", y=error_name, x="graph_mode")
plt.xlabel("")
plt.xticks(rotation=90)
plt.title("Coordinate encoding ablation")
plt.tight_layout()
plt.savefig(f"output/plots/ablation/sonati_ablation_coord_encoding_{error_name}.png")
plt.show()


# Mask information

Ablates whether missingness indicators (binary mask vectors) are fed as input features into the model and/or the feature mixer.

In [ ]:
# Influence of mask indicators as input features
exp_ids = [11, 27, 28, 37, 38, 76, # autoencoder
           0, 1, 3, 15, 25, 30, # cross_attention (plain, use_masks, feat, feat_mask, feat_mask+use_masks, feat+use_masks
           17, 31, 72, 73, 74, 77, # space_time
           90, 245, 246, 247, 255, 319 # time_space
           ]
attr_list = ["attention_type", "feature_mixer_input", "feature_mixer", "use_masks"]
error_name = "RMSE"

# Load experiments
results, df_woa, df_cmip = load_experiments(exp_ids, attr_list, woa=True)
results["model_label"] = results["attention_type"] + np.where(results["feature_mixer"], ", feature mixer", "")
fm = results["feature_mixer"]
fmm = results["feature_mixer_input"] == "feat_mask"
att = results["use_masks"]
results["Masks"] = np.select(
    [fmm & att & fm, fmm & fm, att],
    ["Feature mixer, attention", "Feature mixer", "Attention"],
    default="None"
)
results = results.sort_values("model_label")

# Plot (with feature mixer)
plt.figure(figsize=(7,6))
barplot_seeds(results[results["feature_mixer"]], x="model_label", hue="Masks", y=error_name)
plt.xlabel("")
plt.xticks(rotation=90)
plt.legend(loc="upper right", title="Masks")
plt.tight_layout()
plt.savefig(f"{config.output_dir_plots}/ablation/sonati_ablation_masks_feature_mixer_{error_name}.png")
plt.show()

# Plot (no feature mixer)
barplot_seeds(results[~results["feature_mixer"]], x="model_label", hue="Masks", y=error_name)
plt.xlabel("")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f"{config.output_dir_plots}/ablation/sonati_ablation_masks_no_feature_mixer_{error_name}.png")
plt.show()
